# CUAD-QA Unsloth fine-tune (Llama-3.2-3B)

1. **Runtime → Change runtime type → T4 GPU**
2. Run the install cell.
3. Run the upload cell and choose both files from your Mac:
   - `cuad_qa_train.jsonl.gz`
   - `cuad_qa_valid.jsonl.gz`
   They live in `Research/M&A Contract Exraction Model/.cache/`
4. Run the train cell (about 30–90 minutes on T4).

In [ ]:
import torch
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then Runtime → Restart session"
print("GPU:", torch.cuda.get_device_name(0))
!pip install -q unsloth datasets trl

In [ ]:
from google.colab import files

print("Select cuad_qa_train.jsonl.gz AND cuad_qa_valid.jsonl.gz")
uploaded = files.upload()
print("Got:", list(uploaded))

In [ ]:
import gzip
import json
from pathlib import Path

from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 4096


def load_jsonl_gz(path):
    rows = []
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return Dataset.from_list(rows)


train_files = list(Path(".").glob("**/*train*.jsonl.gz"))
valid_files = list(Path(".").glob("**/*valid*.jsonl.gz"))
assert train_files and valid_files, "Upload both jsonl.gz files in the previous cell."
train_ds = load_jsonl_gz(train_files[0])
valid_ds = load_jsonl_gz(valid_files[0])
print(f"train={len(train_ds)} valid={len(valid_ds)}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)


def formatting_prompts_func(examples):
    texts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        for messages in examples["messages"]
    ]
    return {"text": texts}


train_ds = train_ds.map(formatting_prompts_func, batched=True)
valid_ds = valid_ds.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.05,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
        output_dir="outputs_cuad_qa",
        seed=42,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
trainer.train()
model.save_pretrained("lora_cuad_qa")
tokenizer.save_pretrained("lora_cuad_qa")
print("Saved LoRA adapter to ./lora_cuad_qa")

In [ ]:
from google.colab import files
import shutil

shutil.make_archive("lora_cuad_qa", "zip", "lora_cuad_qa")
files.download("lora_cuad_qa.zip")